## Here we plot some benchmark results. The benchmark were performed on Google Colab CPU or GPU (free tier) in order to be reproducible.

### Useful functions

In [ ]:
import time
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platforms", 'cpu') # put 'gpu' here if you have one
import jax.numpy as jnp

def run_benchmark(func, func_args, func_kwargs, repetition=10):
  # warmuo
  func(*func_args, **func_kwargs)
  times = []
  for _ in range(repetition):
    t0 = time.time()
    res = func(*func_args, **func_kwargs).block_until_ready()
    tf = time.time()
    times.append(tf-t0)
  return jnp.mean(jnp.asarray(times)), jnp.std(jnp.asarray(times))

def jax_gkde_1d(points, data, weights, bw):
  return jax.scipy.stats.gaussian_kde(data,
                                      weights=weights,
                                      bw_method=bw
                                      )(points)

# Univariate case

## 1. Compute everything within KDE FFT

In [ ]:
import os,sys
parent_dir = os.getcwd() + '/../src/'
sys.path.append(parent_dir)
from KDExpress import fft_kde1d, binned_kde1d, silverman_bw1d, build_hist_edges

key = jax.random.PRNGKey(42)
data_1d = jax.random.normal(key, shape=(10_000,))
weights = data_1d**2
points_1d = jnp.linspace(jnp.min(data_1d)-0.5, jnp.max(data_1d)-0.5, 500)

fft_kde_1d_args = [points_1d, data_1d]
fft_kde_1d_kwargs = {'weights': weights}

binned_kde_1d_args = [points_1d, data_1d]
binned_kde_1d_kwargs = {'weights': weights, 'nbins':200}

jax_kde_args = [points_1d, data_1d, weights, silverman_bw1d(data_1d)]
jax_kde_kwargs = {}

# Run the following on Colab for consistency
print(run_benchmark(fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs))
print(run_benchmark(binned_kde1d, binned_kde_1d_args, binned_kde_1d_kwargs))
print(run_benchmark(jax_gkde_1d, jax_kde_args, jax_kde_kwargs))

E0813 10:48:52.740760   28270 pjrt_stream_executor_client.cc:3077] Execution of replica 0 failed: INTERNAL: CustomCall failed: CpuCallback error: Traceback (most recent call last):
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/runpy.py", line 196, in _run_module_as_main
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/runpy.py", line 86, in _run_code
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/home/mt/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start
  File "/home/mt/.local/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/asyncio/base_events.py", line 603, in run_forever
  File "/home/mt/softwares/mini

XlaRuntimeError: INTERNAL: CustomCall failed: CpuCallback error: Traceback (most recent call last):
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/runpy.py", line 196, in _run_module_as_main
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/runpy.py", line 86, in _run_code
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
  File "/home/mt/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 739, in start
  File "/home/mt/.local/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 205, in start
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/asyncio/base_events.py", line 603, in run_forever
  File "/home/mt/softwares/miniconda3/envs/gw-mdc/lib/python3.10/asyncio/base_events.py", line 1909, in _run_onc

## 2. Precompute bin_edges (useful when the grid is fixed)

In [ ]:
be = build_hist_edges(data_1d)
fft_kde_1d_args_2 = [points_1d, data_1d]
fft_kde_1d_kwargs_2 = {'weights': weights, 'bin_edges':be}

# Run the following on Colab for consistency
print(run_benchmark(fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs))

## 3. Case in which points have support much wider than data

In [ ]:
points_1d_wide = jnp.linspace(jnp.min(data_1d)-5*jnp.std(data_1d), jnp.max(data_1d)+5*jnp.std(data_1d), 500)
be = build_hist_edges(points_1d_wide)

fft_kde_1d_args = [points_1d_wide, data_1d]
fft_kde_1d_kwargs = {'weights': weights}
fft_kde_1d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

binned_kde_1d_args = [points_1d_wide, data_1d]
binned_kde_1d_kwargs_no_cut = {'weights': weights, 'nbins':200,}
binned_kde_1d_kwargs_with_cut = {'weights': weights, 'nbins':200, 'cut_sigma_data':2}

jax_kde_args = [points_1d, data_1d, weights, silverman_bw1d(data_1d)]
jax_kde_kwargs = {}

# Run the following on Colab for consistency
print(run_benchmark(fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs))
print(run_benchmark(fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs_with_be))
print(run_benchmark(binned_kde1d, binned_kde_1d_args, binned_kde_1d_kwargs_no_cut))
print(run_benchmark(binned_kde1d, binned_kde_1d_args, binned_kde_1d_kwargs_with_cut))
print(run_benchmark(jax_gkde_1d, jax_kde_args, jax_kde_kwargs))

# 4. Varying points resolution

In [ ]:
fft_kde_1d_args = [points_1d, data_1d]
fft_kde_1d_kwargs = {'weights': weights}
fft_kde_1d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

binned_kde_1d_args = [points_1d, data_1d]
binned_kde_1d_kwargs = {'weights': weights, 'nbins':200,}

jax_kde_args = [points_1d, data_1d, weights, silverman_bw1d(data_1d)]
jax_kde_kwargs = {}

methods = {
  'fft_kde': (fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs),
  'fft_kde_with_be': (fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs_with_be),
  'binned_kde': (binned_kde1d, binned_kde_1d_args, binned_kde_1d_kwargs),
  'jax_kde': (jax_gkde_1d, jax_kde_args, jax_kde_kwargs)
}

def compute_benchmark_point_res(points_res):
  mean_times = defaultdict(list)
  for res in points_res:
    points_1d = jnp.linspace(jnp.min(data_1d)-0.5, jnp.max(data_1d)+0.5, res)
    be = build_hist_edges(points_1d)
    fft_kde_1d_args[0] = points_1d
    binned_kde_1d_args[0] = points_1d
    jax_kde_args[0] = points_1d

    fft_kde_1d_kwargs_with_be['bin_edges'] = be

    print(f"\nResolution: {res} points")
    for method_name, (func, args, kwargs) in methods.items():
      mean, std = run_benchmark(func, args, kwargs)
      mean_times[method_name].append(float(mean))
      print(f"{method_name}: {mean:.5f} ± {std:.5f} s")
  return dict(mean_times)

mean_times_point_res = compute_benchmark_point_res([200,400,600,800,1000])

# 5. Varying sample resolution

In [ ]:
fft_kde_1d_args = [points_1d, data_1d]
fft_kde_1d_kwargs = {'weights': weights}
fft_kde_1d_kwargs_with_be = {'weights': weights, 'bin_edges':be}

binned_kde_1d_args = [points_1d, data_1d]
binned_kde_1d_kwargs = {'weights': weights, 'nbins':200,}

jax_kde_args = [points_1d, data_1d, weights, silverman_bw1d(data_1d)]
jax_kde_kwargs = {}

methods = {
  'fft_kde': (fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs),
  'fft_kde_with_be': (fft_kde1d, fft_kde_1d_args, fft_kde_1d_kwargs_with_be),
  'binned_kde': (binned_kde1d, binned_kde_1d_args, binned_kde_1d_kwargs),
  'jax_kde': (jax_gkde_1d, jax_kde_args, jax_kde_kwargs)
}

def compute_benchmark_data_res(data_res):
  mean_times = defaultdict(list)
  for res in data_res:
    key = jax.random.PRNGKey(42)
    data_1d = jax.random.normal(key, shape=(res,))
    weights = data_1d**2
    points_1d = jnp.linspace(jnp.min(data_1d)-0.5, jnp.max(data_1d)+0.5, 500)
    be = build_hist_edges(points_1d)

    fft_kde_1d_args[0] = points_1d
    binned_kde_1d_args[0] = points_1d
    fft_kde_1d_args[1] = data_1d
    binned_kde_1d_args[1] = data_1d

    fft_kde_1d_kwargs['weights'] = weights
    binned_kde_1d_kwargs['weights'] = weights
    fft_kde_1d_kwargs_with_be['weights'] = weights
    fft_kde_1d_kwargs_with_be['bin_edges'] = be

    jax_kde_args[0] = points_1d
    jax_kde_args[1] = data_1d
    jax_kde_args[2] = weights
    jax_kde_args[3] = silverman_bw1d(data_1d)


    print(f"\nResolution: {res} samples")
    for method_name, (func, args, kwargs) in methods.items():
      mean, std = run_benchmark(func, args, kwargs)
      mean_times[method_name].append(float(mean))
      print(f"{method_name}: {mean:.5f} ± {std:.5f} s")
  return dict(mean_times)

data_res = [ 5_000, 7_500, 10_000, 12_500, 15_000]
mean_times_data_res = compute_benchmark_data_res(data_res)

# Final plots

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with two subplots side by side with shared y-axis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# First subplot - Points Resolution
for method in methods:
    ax1.plot(points_res, mean_times_point_res[method], marker='o', label=method)
ax1.set_xlabel('Points Resolution [Data = (10_000,)]')
ax1.set_ylabel('Mean Execution Time (s)')
ax1.set_yscale('log')
ax1.set_ylim(1e-3, 2e-1)
ax1.set_title('CPU Times')
ax1.legend()
ax1.grid(True)

# Second subplot - Data Resolution
for method in methods:
    ax2.plot(data_res, mean_times_data_res[method], marker='o', label=method)
ax2.set_xlabel('Data Resolution [Points = (500,)]')
ax2.set_title('CPU Times')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('cpu_bench_1d.png', dpi=600)
plt.show()